# M1_00 — All-in-one extraction (1D + 1H + external 1D + external 1H)

Objetivo (M1):
- Descargar OHLCV 1D y 1H desde CoinDesk Data API.
- Generar datasets raw + clean + QA.
- Descargar features externas 1D (macro + FGI) y replicarlas a 1H (forward-fill intradía).

Qué dataset se usa para modelado:
- Este notebook NO genera el dataset final de entrenamiento.
- El dataset final para modelos 1D sale de M2 y se llama explícitamente:
  - `btc_1d_features__base.csv` (baseline)
  - `btc_1d_features__tech.csv` (recomendado por defecto)
  - `btc_1d_features__macro.csv`
  - `btc_1d_features__sent.csv`
  - `btc_1d_features__full.csv`

Outputs:
- Inputs (para M2) en `../data/_inputs/`:
  - `xbx_coindesk_ohlcv_1d_{raw,clean}.csv`
  - `xbx_coindesk_ohlcv_1h_{raw,clean}.csv`
  - `external_1d__{macro,sent}.csv`
  - `external_1h__{macro,sent}.csv`
- QA en `../data/_qa/`:
  - `data_quality__xbx_1d.json`, `data_quality__xbx_1h.json`
  - `data_quality__external_1d.json`, `data_quality__external_1h.json`


In [1]:
import os
import sys
import json
import pathlib
import importlib
import subprocess

for p in ["pandas", "numpy", "requests"]:
    try:
        importlib.import_module(p)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", p])

try:
    importlib.import_module("yfinance")
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "yfinance"])

import numpy as np
import pandas as pd
import requests
import yfinance as yf


In [2]:
market = "sda"
instrument = "XBX-USD"
limit = 2000

base_dir = pathlib.Path(".").resolve()
data_dir = (base_dir.parent / "data").resolve()
data_dir.mkdir(parents=True, exist_ok=True)
inputs_dir = (data_dir / "_inputs").resolve()
qa_dir = (data_dir / "_qa").resolve()
inputs_dir.mkdir(parents=True, exist_ok=True)
qa_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "raw_1d": inputs_dir / "xbx_coindesk_ohlcv_1d_raw.csv",
    "clean_1d": inputs_dir / "xbx_coindesk_ohlcv_1d_clean.csv",
    "qa_1d": qa_dir / "data_quality__xbx_1d.json",
    "raw_1h": inputs_dir / "xbx_coindesk_ohlcv_1h_raw.csv",
    "clean_1h": inputs_dir / "xbx_coindesk_ohlcv_1h_clean.csv",
    "qa_1h": qa_dir / "data_quality__xbx_1h.json",
    "external_1d_macro": inputs_dir / "external_1d__macro.csv",
    "external_1d_sent": inputs_dir / "external_1d__sent.csv",
    "qa_external_1d": qa_dir / "data_quality__external_1d.json",
    "external_1h_macro": inputs_dir / "external_1h__macro.csv",
    "external_1h_sent": inputs_dir / "external_1h__sent.csv",
    "qa_external_1h": qa_dir / "data_quality__external_1h.json",
}

{k: v.name for k, v in paths.items()}

{'raw_1d': 'xbx_coindesk_ohlcv_1d_raw.csv',
 'clean_1d': 'xbx_coindesk_ohlcv_1d_clean.csv',
 'qa_1d': 'data_quality__xbx_1d.json',
 'raw_1h': 'xbx_coindesk_ohlcv_1h_raw.csv',
 'clean_1h': 'xbx_coindesk_ohlcv_1h_clean.csv',
 'qa_1h': 'data_quality__xbx_1h.json',
 'external_1d_macro': 'external_1d__macro.csv',
 'external_1d_sent': 'external_1d__sent.csv',
 'qa_external_1d': 'data_quality__external_1d.json',
 'external_1h_macro': 'external_1h__macro.csv',
 'external_1h_sent': 'external_1h__sent.csv',
 'qa_external_1h': 'data_quality__external_1h.json'}

In [3]:
datasets_for_models_1d = [
    data_dir / "btc_1d_features__tech.csv",
    data_dir / "btc_1d_features__full.csv",
    data_dir / "btc_1d_features__macro.csv",
    data_dir / "btc_1d_features__sent.csv",
    data_dir / "btc_1d_features__base.csv",
]

print("M1 outputs (inputs a M2):")
print("-", paths["clean_1d"].name)
print("-", paths["external_1d_macro"].name)
print("-", paths["external_1d_sent"].name)
print("-", paths["clean_1h"].name)

print("\nDatasets finales para modelos 1D (M2):")
for p in datasets_for_models_1d:
    print("-", p.name, "exists=" + str(p.exists()))

default_model_dataset_1d = datasets_for_models_1d[0]
print("\nDEFAULT_MODEL_DATASET_1D:")
print(default_model_dataset_1d.as_posix())


M1 outputs (inputs a M2):
- xbx_coindesk_ohlcv_1d_clean.csv
- external_1d__macro.csv
- external_1d__sent.csv
- xbx_coindesk_ohlcv_1h_clean.csv

Datasets finales para modelos 1D (M2):
- btc_1d_features__tech.csv exists=True
- btc_1d_features__full.csv exists=True
- btc_1d_features__macro.csv exists=True
- btc_1d_features__sent.csv exists=True
- btc_1d_features__base.csv exists=True

DEFAULT_MODEL_DATASET_1D:
/Users/williamvasquez/Library/CloudStorage/OneDrive-Personal/Documentos/William/cursos Online/Masters/IA VIU/trabajo fin master/proyecto_grado/data/btc_1d_features__tech.csv


In [4]:
BASE_URL = "https://data-api.coindesk.com"

def load_env_file(path: str = ".env"):
    p = pathlib.Path(path)
    if not p.exists():
        return
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if (not line) or line.startswith("#") or ("=" not in line):
            continue
        k, v = line.split("=", 1)
        os.environ.setdefault(k.strip(), v.strip())


load_env_file(".env")

API_KEY = os.getenv("COINDESK_API_KEY") or os.getenv("COINDESK_APIKEY") or os.getenv("COINDESK_KEY")
SESSION = requests.Session()
HEADERS = {}
if API_KEY:
    HEADERS["Authorization"] = f"Apikey {API_KEY}"
    HEADERS["x-api-key"] = API_KEY

bool(API_KEY)

False

In [5]:
def fetch_market_instrument_metadata():
    url = f"{BASE_URL}/index/cc/v1/markets/instruments"
    params = {"market": market, "instruments": instrument, "instrument_status": "ACTIVE"}
    r = SESSION.get(url, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    payload = r.json()
    data = payload.get("Data") or payload.get("data")
    if isinstance(data, dict):
        market_node = data.get(market)
        if isinstance(market_node, dict):
            instruments = market_node.get("instruments") or market_node.get("Instruments")
            if isinstance(instruments, dict):
                row = instruments.get(instrument) or instruments.get(instrument.replace("-", ""))
                if isinstance(row, dict):
                    return row
        row = data.get(instrument)
        if isinstance(row, dict):
            return row
    if isinstance(data, list):
        for row in data:
            if row.get("INSTRUMENT") == instrument or row.get("instrument") == instrument:
                return row
    return None


def fetch_endpoint(endpoint: str, to_ts=None):
    url = f"{BASE_URL}{endpoint}"
    params = {"market": market, "instrument": instrument, "limit": int(limit), "groups": "OHLC,VOLUME"}
    if to_ts is not None:
        params["to_ts"] = int(to_ts)
    r = SESSION.get(url, params=params, headers=HEADERS, timeout=30)
    try:
        payload = r.json()
    except Exception:
        payload = {"Data": [], "Err": {"message": r.text}}
    payload["_http_status"] = r.status_code
    if r.status_code >= 400 and r.status_code != 404:
        r.raise_for_status()
    return payload


def download_all(endpoint: str, step_seconds: int, max_pages: int, oldest_meta_key: str | None):
    meta = fetch_market_instrument_metadata()
    oldest_allowed = None
    if meta:
        cand = None
        if oldest_meta_key is not None:
            cand = meta.get(oldest_meta_key)
        cand = cand or meta.get("OLDEST_HISTORICAL_DAY_DATA_TIMESTAMP") or meta.get("FIRST_INDEX_UPDATE_TIMESTAMP")
        if cand is not None:
            oldest_allowed = int(cand)

    all_rows = []
    to_ts = None
    for _ in range(int(max_pages)):
        payload = fetch_endpoint(endpoint, to_ts=to_ts)
        if payload.get("_http_status") == 404:
            break
        data = payload.get("Data") or payload.get("data") or []
        if not data:
            break
        all_rows.extend(data)
        ts_vals = [d.get("TIMESTAMP") for d in data if d.get("TIMESTAMP") is not None]
        if not ts_vals:
            break
        oldest = int(min(ts_vals))
        if oldest_allowed is not None and oldest <= oldest_allowed:
            break
        next_to = oldest - int(step_seconds)
        if to_ts is not None and next_to >= to_ts:
            break
        to_ts = next_to

    df = pd.DataFrame(all_rows)
    return df, meta


def standardize_ohlcv(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    col_map = {
        "TIMESTAMP": "timestamp",
        "OPEN": "open",
        "HIGH": "high",
        "LOW": "low",
        "CLOSE": "close",
        "VOLUME": "volume",
        "QUOTE_VOLUME": "quote_volume",
    }
    for k, v in col_map.items():
        if k in df.columns:
            df = df.rename(columns={k: v})

    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

    for c in ["open", "high", "low", "close", "volume", "quote_volume"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(subset=["timestamp", "open", "high", "low", "close", "volume"])
    df = df.drop_duplicates(subset=["timestamp"], keep="last").sort_values("timestamp").reset_index(drop=True)
    return df


def candle_invalid_count(df: pd.DataFrame) -> int:
    o = df["open"]
    h = df["high"]
    l = df["low"]
    c = df["close"]
    invalid = (h < np.maximum(o, c)) | (l > np.minimum(o, c)) | (h < l)
    return int(invalid.sum())


def compute_continuity(df: pd.DataFrame, freq: str):
    if len(df) == 0:
        return [], 0
    idx = df["timestamp"].dt.floor(freq)
    expected = pd.date_range(start=idx.min(), end=idx.max(), freq=freq, tz="UTC")
    missing = expected.difference(pd.DatetimeIndex(idx.unique()).sort_values())
    missing_list = [d.isoformat() for d in missing.to_pydatetime()]
    return missing_list, int(len(missing_list))


In [6]:
df_1d_raw, meta_1d = download_all(
    endpoint="/index/cc/v1/historical/days",
    step_seconds=86400,
    max_pages=400,
    oldest_meta_key="OLDEST_HISTORICAL_DAY_DATA_TIMESTAMP",
)
df_1d_clean = standardize_ohlcv(df_1d_raw)

df_1d_raw.to_csv(paths["raw_1d"], index=False)
df_1d_clean.to_csv(paths["clean_1d"], index=False)

missing_days, missing_days_count = compute_continuity(df_1d_clean, "D")
qa_1d = {
    "market": market,
    "instrument": instrument,
    "frequency": "1D",
    "raw_rows": int(len(df_1d_raw)),
    "clean_rows": int(len(df_1d_clean)),
    "start": (df_1d_clean["timestamp"].min().isoformat() if len(df_1d_clean) else None),
    "end": (df_1d_clean["timestamp"].max().isoformat() if len(df_1d_clean) else None),
    "missing_days_count": missing_days_count,
    "missing_days_sample": missing_days[:20],
    "negative_volume_count": int((df_1d_clean["volume"] < 0).sum()),
    "invalid_candles_count": candle_invalid_count(df_1d_clean),
    "nan_fraction_by_col": {k: float(v) for k, v in df_1d_clean.isna().mean().to_dict().items()},
    "meta": (meta_1d if isinstance(meta_1d, dict) else None),
}
paths["qa_1d"].write_text(json.dumps(qa_1d, ensure_ascii=False, indent=2), encoding="utf-8")

(df_1d_clean.shape, qa_1d["start"], qa_1d["end"])

((4151, 8), '2014-11-03T00:00:00+00:00', '2026-03-15T00:00:00+00:00')

In [7]:
df_1h_raw, meta_1h = download_all(
    endpoint="/index/cc/v1/historical/hours",
    step_seconds=3600,
    max_pages=2000,
    oldest_meta_key="OLDEST_HISTORICAL_HOUR_DATA_TIMESTAMP",
)
df_1h_clean = standardize_ohlcv(df_1h_raw)

df_1h_raw.to_csv(paths["raw_1h"], index=False)
df_1h_clean.to_csv(paths["clean_1h"], index=False)

missing_hours, missing_hours_count = compute_continuity(df_1h_clean, "h")
qa_1h = {
    "market": market,
    "instrument": instrument,
    "frequency": "1H",
    "raw_rows": int(len(df_1h_raw)),
    "clean_rows": int(len(df_1h_clean)),
    "start": (df_1h_clean["timestamp"].min().isoformat() if len(df_1h_clean) else None),
    "end": (df_1h_clean["timestamp"].max().isoformat() if len(df_1h_clean) else None),
    "missing_hours_count": missing_hours_count,
    "missing_hours_sample": missing_hours[:20],
    "negative_volume_count": int((df_1h_clean["volume"] < 0).sum()),
    "invalid_candles_count": candle_invalid_count(df_1h_clean),
    "nan_fraction_by_col": {k: float(v) for k, v in df_1h_clean.isna().mean().to_dict().items()},
    "meta": (meta_1h if isinstance(meta_1h, dict) else None),
}
paths["qa_1h"].write_text(json.dumps(qa_1h, ensure_ascii=False, indent=2), encoding="utf-8")

(df_1h_clean.shape, qa_1h["start"], qa_1h["end"])

((99611, 8), '2014-11-03T00:00:00+00:00', '2026-03-15T10:00:00+00:00')

In [8]:
btc_days = pd.DatetimeIndex(pd.to_datetime(df_1d_clean["timestamp"], utc=True).dt.floor("D"), tz="UTC")

tickers = {
    "^GSPC": "sp500",
    "DX-Y.NYB": "dxy",
    "^VIX": "vix",
    "GC=F": "gold"
}

start_date = (btc_days.min() - pd.Timedelta(days=7)).date().isoformat()
end_date = (btc_days.max() + pd.Timedelta(days=1)).date().isoformat()

macro_frames = {}
for ticker, name in tickers.items():
    dfm = yf.download(ticker, start=start_date, end=end_date, progress=False)
    if dfm is None or len(dfm) == 0:
        macro_frames[name] = pd.DataFrame(index=btc_days, columns=[name], dtype=float)
        continue
    dfm = dfm[["Close"]].copy()
    dfm.columns = [name]
    idx = pd.to_datetime(dfm.index)
    dfm.index = pd.DatetimeIndex(idx.date, tz="UTC")
    dfm = dfm[~dfm.index.duplicated(keep="last")].sort_index()
    dfm = dfm.reindex(btc_days)
    dfm[name] = dfm[name].ffill(limit=7)
    macro_frames[name] = dfm

df_macro = pd.DataFrame(index=btc_days)
for name, dfm in macro_frames.items():
    df_macro[name] = pd.to_numeric(dfm[name], errors="coerce")
    df_macro[f"log_ret_{name}"] = np.log(df_macro[name] / df_macro[name].shift(1))
    df_macro[f"vol_7d_{name}"] = df_macro[f"log_ret_{name}"].rolling(window=7).std()

df_macro = df_macro.reset_index().rename(columns={"index": "timestamp"})
df_macro["timestamp"] = pd.to_datetime(df_macro["timestamp"], utc=True).dt.strftime("%Y-%m-%d %H:%M:%S+00:00")
df_macro.to_csv(paths["external_1d_macro"], index=False)

df_macro.head(3)

,timestamp,sp500,log_ret_sp500,vol_7d_sp500,dxy,log_ret_dxy,vol_7d_dxy,vix,log_ret_vix,vol_7d_vix,gold,log_ret_gold,vol_7d_gold
0,2014-11-03 00:00:00+00:00,2017.810059,NaN,NaN,87.309998,NaN,NaN,14.73,NaN,NaN,1169.400024,NaN,NaN
1,2014-11-04 00:00:00+00:00,2012.099976,-0.002834,NaN,86.980003,-0.003787,NaN,14.89,0.010804,NaN,1167.400024,-0.001712,NaN
2,2014-11-05 00:00:00+00:00,2023.569946,0.005684,NaN,87.440002,0.005275,NaN,14.17,-0.049563,NaN,1145.400024,-0.019025,NaN


In [9]:
def fetch_fgi_history():
    url = "https://api.alternative.me/fng/"
    params = {"limit": 0, "format": "json", "date_format": "us"}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json().get("data") or []
    df = pd.DataFrame(data)
    if len(df) == 0:
        return pd.DataFrame(columns=["timestamp", "fgi", "fgi_norm"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%m-%d-%Y", errors="coerce")
    df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize("UTC")
    df["fgi"] = pd.to_numeric(df["value"], errors="coerce")
    df = df[["timestamp", "fgi"]].dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)
    df["fgi_norm"] = df["fgi"] / 100.0
    df["timestamp"] = df["timestamp"].dt.floor("D")
    df = df.drop_duplicates(subset=["timestamp"], keep="last")
    return df


df_fgi = fetch_fgi_history()
df_sent_1d = pd.DataFrame({"timestamp": btc_days}).merge(df_fgi, on="timestamp", how="left")
df_sent_1d["timestamp"] = pd.to_datetime(df_sent_1d["timestamp"], utc=True).dt.strftime("%Y-%m-%d %H:%M:%S+00:00")
df_sent_1d.to_csv(paths["external_1d_sent"], index=False)

df_sent_1d.tail(3)

,timestamp,fgi,fgi_norm
4148,2026-03-13 00:00:00+00:00,15.0,0.15
4149,2026-03-14 00:00:00+00:00,16.0,0.16
4150,2026-03-15 00:00:00+00:00,15.0,0.15


In [10]:
qa_external_1d = {
    "btc_calendar_start": pd.to_datetime(btc_days.min()).isoformat(),
    "btc_calendar_end": pd.to_datetime(btc_days.max()).isoformat(),
    "macro_rows": int(len(df_macro)),
    "sent_rows": int(len(df_sent_1d)),
    "macro_missing_fraction": {k: float(v) for k, v in pd.read_csv(paths["external_1d_macro"]).isna().mean().to_dict().items()},
    "sent_missing_fraction": {k: float(v) for k, v in pd.read_csv(paths["external_1d_sent"]).isna().mean().to_dict().items()},
}
paths["qa_external_1d"].write_text(json.dumps(qa_external_1d, ensure_ascii=False, indent=2), encoding="utf-8")
paths["qa_external_1d"].as_posix()

'/Users/williamvasquez/Library/CloudStorage/OneDrive-Personal/Documentos/William/cursos Online/Masters/IA VIU/trabajo fin master/proyecto_grado/data/_qa/data_quality__external_1d.json'

In [11]:
df_1h = pd.read_csv(paths["clean_1h"])
df_1h["timestamp"] = pd.to_datetime(df_1h["timestamp"], utc=True)
df_1h = df_1h.sort_values("timestamp").reset_index(drop=True)

hour_index = pd.DatetimeIndex(df_1h["timestamp"].dt.floor("h"), tz="UTC")
day_index = pd.DatetimeIndex(df_1h["timestamp"].dt.floor("D"), tz="UTC")

df_base = pd.DataFrame({"timestamp": hour_index})
df_base["day"] = day_index

macro_1d = pd.read_csv(paths["external_1d_macro"])
macro_1d["timestamp"] = pd.to_datetime(macro_1d["timestamp"], utc=True).dt.floor("D")
macro_1d = macro_1d.sort_values("timestamp").drop_duplicates(subset=["timestamp"], keep="last").rename(columns={"timestamp": "day"})

sent_1d = pd.read_csv(paths["external_1d_sent"])
sent_1d["timestamp"] = pd.to_datetime(sent_1d["timestamp"], utc=True).dt.floor("D")
sent_1d = sent_1d.sort_values("timestamp").drop_duplicates(subset=["timestamp"], keep="last").rename(columns={"timestamp": "day"})

macro_cols = [c for c in macro_1d.columns if c != "day"]
sent_cols = [c for c in sent_1d.columns if c != "day"]

df_macro_1h = df_base[["timestamp", "day"]].merge(macro_1d, on="day", how="left").drop(columns=["day"]) 
df_sent_1h = df_base[["timestamp", "day"]].merge(sent_1d, on="day", how="left").drop(columns=["day"]) 

if "fgi" in df_sent_1h.columns:
    df_sent_1h["fgi"] = pd.to_numeric(df_sent_1h["fgi"], errors="coerce")
    df_sent_1h["fgi_ffill50"] = df_sent_1h["fgi"].ffill().fillna(50)
    df_sent_1h["fgi_norm_ffill50"] = df_sent_1h["fgi_ffill50"] / 100.0
    sent_cols = list(sent_cols) + ["fgi_ffill50", "fgi_norm_ffill50"]

df_macro_1h["timestamp"] = pd.to_datetime(df_macro_1h["timestamp"], utc=True).dt.strftime("%Y-%m-%d %H:%M:%S+00:00")
df_sent_1h["timestamp"] = pd.to_datetime(df_sent_1h["timestamp"], utc=True).dt.strftime("%Y-%m-%d %H:%M:%S+00:00")

df_macro_1h.to_csv(paths["external_1h_macro"], index=False)
df_sent_1h.to_csv(paths["external_1h_sent"], index=False)

(df_macro_1h.shape, df_sent_1h.shape)

((99611, 13), (99611, 5))

In [12]:
qa_external_1h = {
    "calendar_1h_start": pd.to_datetime(hour_index.min()).isoformat(),
    "calendar_1h_end": pd.to_datetime(hour_index.max()).isoformat(),
    "macro_cols": macro_cols,
    "sent_cols": sent_cols,
    "macro_rows": int(len(df_macro_1h)),
    "sent_rows": int(len(df_sent_1h)),
    "macro_missing_fraction": {k: float(v) for k, v in pd.read_csv(paths["external_1h_macro"]).isna().mean().to_dict().items()},
    "sent_missing_fraction": {k: float(v) for k, v in pd.read_csv(paths["external_1h_sent"]).isna().mean().to_dict().items()},
}
paths["qa_external_1h"].write_text(json.dumps(qa_external_1h, ensure_ascii=False, indent=2), encoding="utf-8")
paths["qa_external_1h"].as_posix()

'/Users/williamvasquez/Library/CloudStorage/OneDrive-Personal/Documentos/William/cursos Online/Masters/IA VIU/trabajo fin master/proyecto_grado/data/_qa/data_quality__external_1h.json'